# Commercial Performance & Promotion Intelligence

## tl;dr

Synthetic net revenue is **€12,551,043**, gross margin is **61.5%**, and forecast WAPE is **2.2%**. The strongest promotion band by weighted scenario ROI is **10%**. **West** has the weakest budget variance and is the first review segment.


## Context & Methods

Decision: where should a fictional commercial team reallocate promotion investment while protecting margin? Data is synthetic and generated with seed 41001. Promotion ROI uses a model-derived baseline and is not a causal estimate.

### Key Assumptions

- EUR is the reporting currency.
- Returns reduce units and revenue.
- The locked forecast uses calendar, product, market, channel, seasonality and known promotion inputs; it is generated before the month and excludes realized daily demand noise.
- Promotion scenarios combine category price response with a disclosed mechanic-specific execution lift; every discount band spans all four mechanics.


In [1]:
from pathlib import Path
import csv, json
PROJECT_DIR = Path.cwd()
def read_csv(name):
    with (PROJECT_DIR / name).open(encoding='utf-8') as handle:
        return list(csv.DictReader(handle))
summary = json.loads((PROJECT_DIR / 'data/curated/summary.json').read_text())
monthly = read_csv('data/curated/monthly_performance.csv')
promos = read_csv('data/curated/promotion_performance.csv')
print(f"Snapshot: 2026-06-30 | months={len(monthly)} | promotion bands={len(promos)}")
print(json.dumps(summary, indent=2))


Snapshot: 2026-06-30 | months=18 | promotion bands=4
{
  "net_revenue": 12551042.74,
  "budget_variance_rate": -0.00934,
  "gross_margin_rate": 0.615324,
  "net_units": 4157892,
  "forecast_wape": 0.022167
}


## Data

The modeled grain is one row per date × SKU × market × channel. Curated files contain only reviewed aggregates used by the dashboard.


In [2]:
from collections import Counter
with (PROJECT_DIR / 'data/raw/sales_daily.csv').open(encoding='utf-8') as handle:
    reader = csv.DictReader(handle)
    row_count = 0
    categories = Counter()
    revenue = 0.0
    gross_profit = 0.0
    for row in reader:
        row_count += 1
        categories[row['category']] += 1
        revenue += float(row['net_revenue_eur'])
        gross_profit += float(row['gross_profit_eur'])
print(f"rows={row_count:,} | categories={dict(categories)}")
print(f"recomputed revenue=€{revenue:,.2f} | margin={gross_profit/revenue:.2%}")


rows=139,776 | categories={'Hydration': 34944, 'Breakfast': 34944, 'Snacks': 34944, 'Plant-Based': 34944}
recomputed revenue=€12,551,042.74 | margin=61.53%


## Results

Promotion bands are compared with weighted totals so small promotions do not receive the same influence as large investments.


In [3]:
ranked = sorted(promos, key=lambda row: float(row['promotion_roi']), reverse=True)
for row in ranked:
    print(f"{row['discount_band']:>3} | ROI {float(row['promotion_roi']):>7.1%} | uplift {float(row['unit_uplift_pct']):>7.1%} | investment €{float(row['promotion_investment_eur']):,.0f}")


10% | ROI   24.5% | uplift   45.4% | investment €7,283
15% | ROI    7.9% | uplift   57.8% | investment €9,661
20% | ROI    1.4% | uplift   71.5% | investment €12,886
30% | ROI  -41.6% | uplift   54.6% | investment €18,406


## Takeaways

1. **Promotion decision:** Run a bounded next-quarter test in the 10% band (modeled weighted ROI 24.5%); keep the modeled baseline and margin guardrail visible before scaling.
2. Review **West** first because its synthetic budget variance is the weakest, not because it has the smallest revenue.
3. Use the product table to distinguish price/discount pressure from a genuine volume shortfall.
